# Data Cleaning
Clean and filter all three datasets, then save to `data/2_cleaned/`.
1. LGBT_EU:   or  
2. HIV_AIDS: 
3. UNICEF_Immunization: 


| Dataset | Files | Notes |
|---|---|---|
| `lgbt_EU` | 4 CSVs | A study on European Queer adults on various life experiences, like experiencing bigotry. The foundation of this analysis|
| `HIV_AIDS_data` | 6 CSVs (two schemas) | A study on HIV and AIDS prevalence, survivors, deaths, and experiences. Worldwide, will be filtered to EU only |
| `UNICEF_Immunization` | 1 xlsx, many vaccine sheets | A record of rates of immunization on a wide range of various vaccines.  Worldwide, will be filtered to EU only |

**Prerequisite:** `5_download_dataset.ipynb` — all raw files must exist in `data/1_source/`.

**Output:** `data/2_cleaned/` with one CSV per dataset (HIV/AIDS split by schema type).

## Setup

In [16]:
# !pip install -r ../requirements.txt


In [17]:
import sys
import re
from pathlib import Path

import pandas as pd

In [18]:
# Ensure src/ is on the path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_download import *

CLEANED_DIR = PROJECT_ROOT / "data" / "2_cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Source dir   : {SOURCE_DIR}")
print(f"Cleaned dir  : {CLEANED_DIR}")

Project root : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
Source dir   : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\1_source
Cleaned dir  : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\2_cleaned


---
## 1. LGBT EU Survey

**Goals:**
- Load 4 CSVs (skip `SubsetSize`)
- Drop the `notes` column if present
- Remove rows where `CountryCode == 'Average'`
- Extract the canonical EU country list (used to filter the other datasets)
- Save one cleaned CSV per source file

In [19]:
LGBT_DIR = SOURCE_DIR / "lgbt_EU"

# Files to process — explicitly exclude SubsetSize
SKIP_LGBT = {"LGBT_Survey_SubsetSize.csv"}

lgbt_files = [
    f for f in sorted(LGBT_DIR.glob("*.csv"))
    if f.name not in SKIP_LGBT
]

print(f"LGBT files to clean: {len(lgbt_files)}")
for f in lgbt_files:
    print(f"  {f.name}")

LGBT files to clean: 5
  LGBT_Survey_DailyLife.csv
  LGBT_Survey_Discrimination.csv
  LGBT_Survey_RightsAwareness.csv
  LGBT_Survey_TransgenderSpecificQuestions.csv
  LGBT_Survey_ViolenceAndHarassment.csv


In [20]:
def clean_lgbt_csv(path):
    """
    Load and clean one LGBT survey CSV.
    - Drops 'notes' column if present
    - Removes rows where CountryCode == 'Average'
    - Strips leading/trailing whitespace from string columns
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape      : {df.shape}")
    print(f"  Columns        : {list(df.columns)}")

    # Drop 'notes' column
    if "notes" in df.columns:
        df = df.drop(columns=["notes"])
        print(f"  Dropped        : 'notes'")

    # Remove 'Average' rows
    before = len(df)
    df = df[df["CountryCode"] != "Average"]
    removed = before - len(df)
    print(f"  Removed 'Average' rows : {removed}")

    # Strip whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

    print(f"  Clean shape    : {df.shape}")
    return df

In [21]:
lgbt_cleaned = {}

for f in lgbt_files:
    key = f.stem   # e.g. 'LGBT_Survey_ViolenceAndHarassment'
    lgbt_cleaned[key] = clean_lgbt_csv(f)


LGBT_Survey_DailyLife.csv
  Raw shape      : (34020, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 1184
  Clean shape    : (32836, 6)

LGBT_Survey_Discrimination.csv
  Raw shape      : (15775, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 547
  Clean shape    : (15228, 6)


C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns
C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m


LGBT_Survey_RightsAwareness.csv
  Raw shape      : (3770, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 130
  Clean shape    : (3640, 6)

LGBT_Survey_TransgenderSpecificQuestions.csv
  Raw shape      : (3421, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 120
  Clean shape    : (3301, 6)

LGBT_Survey_ViolenceAndHarassment.csv
  Raw shape      : (45355, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 1722
  Clean shape    : (43633, 6)


C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns
C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

In [22]:
# Extract canonical EU country list from the survey data
# Use the first cleaned dataframe — all share the same CountryCode values
_sample_df = next(iter(lgbt_cleaned.values()))
EU_COUNTRIES: set[str] = set(_sample_df["CountryCode"].dropna().unique())

print(f"EU countries in survey ({len(EU_COUNTRIES)}):")
print(sorted(EU_COUNTRIES))

EU countries in survey (28):
['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'United Kingdom']


In [23]:
# Inspect a sample of one cleaned dataframe
key = list(lgbt_cleaned.keys())[0]
print(f"Sample from: {key}")
lgbt_cleaned[key].head(6)

Sample from: LGBT_Survey_DailyLife


,CountryCode,subset,question_code,question_label,answer,percentage
0,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very widespread,8
1,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly widespread,34
2,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly rare,45
3,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very rare,9
4,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Don`t know,4
5,Austria,Gay,b1_a,"In your opinion, how widespread is offensive l...",Very widespread,4


In [24]:
# Save cleaned LGBT CSVs
lgbt_out_dir = CLEANED_DIR / "lgbt_EU"
lgbt_out_dir.mkdir(parents=True, exist_ok=True)

for key, df in lgbt_cleaned.items():
    out_path = lgbt_out_dir / f"{key}_cleaned.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"  Saved: {out_path.name}  ({len(df)} rows)")

  Saved: LGBT_Survey_DailyLife_cleaned.csv  (32836 rows)
  Saved: LGBT_Survey_Discrimination_cleaned.csv  (15228 rows)
  Saved: LGBT_Survey_RightsAwareness_cleaned.csv  (3640 rows)
  Saved: LGBT_Survey_TransgenderSpecificQuestions_cleaned.csv  (3301 rows)
  Saved: LGBT_Survey_ViolenceAndHarassment_cleaned.csv  (43633 rows)


In [25]:
LIST_OF_EU_COUNTRIES = [
    'Austria',
    'Belgium',
    'Bulgaria',
    'Croatia', 
    'Cyprus', 
    'Czech Republic', 
    'Denmark', 
    'Estonia', 
    'Finland', 
    'France', 
    'Germany', 
    'Greece', 
    'Hungary', 
    'Ireland', 
    'Italy', 
    'Latvia', 
    'Lithuania', 
    'Luxembourg', 
    'Malta', 
    'Netherlands', 
    'Poland', 
    'Portugal', 
    'Romania', 
    'Slovakia', 
    'Slovenia', 
    'Spain', 
    'Sweden', 
    'United Kingdom'
]

---
## 2. HIV/AIDS Data

Note, there are 2 different formats/schemas in this dataset:

**Schema A** — `no_of_<...>` files:  
Columns: `Country, Year, Count, Count_median, Count_min, Count_max, WHO Region`  
→ Drop `Count` since it uses a bracketed range string, keep `Count_median` as canonical.  
→ Normalize string nulls (`na`, `Na`, etc.) → `NaN`.

**Schema B** — `art_coverage_<...>` files:  
Columns: wide descriptive names, no `Year`, `Nodata` as string null, bracketed ranges in cells.  
→ Normalize `Nodata` / `No data` → `NaN`.  
→ Parse bracketed ranges (e.g. `500[500-500]`) — extract the leading number as the canonical value.

Both: filter to EU countries only.

In [26]:
HIV_DIR = SOURCE_DIR / "HIV_AIDS_data"

hiv_files = sorted(HIV_DIR.glob("*.csv"))
print(f"HIV/AIDS files: {len(hiv_files)}")
for f in hiv_files:
    print(f"  {f.name}")

HIV/AIDS files: 6
  art_coverage_by_country_clean.csv
  art_pediatric_coverage_by_country_clean.csv
  no_of_cases_adults_15_to_49_by_country_clean.csv
  no_of_deaths_by_country_clean.csv
  no_of_people_living_with_hiv_by_country_clean.csv
  prevention_of_mother_to_child_transmission_by_country_clean.csv


In [27]:
# Identify schema by filename prefix
SCHEMA_A_PREFIX = "no_of_"
SCHEMA_B_PREFIX = "art_"

schema_a_files = [f for f in hiv_files if f.name.startswith(SCHEMA_A_PREFIX)]
schema_b_files = [f for f in hiv_files if f.name.startswith(SCHEMA_B_PREFIX)]

print(f"Schema A (no_of_*): {[f.name for f in schema_a_files]}")
print(f"Schema B (art_*  ): {[f.name for f in schema_b_files]}")

Schema A (no_of_*): ['no_of_cases_adults_15_to_49_by_country_clean.csv', 'no_of_deaths_by_country_clean.csv', 'no_of_people_living_with_hiv_by_country_clean.csv']
Schema B (art_*  ): ['art_coverage_by_country_clean.csv', 'art_pediatric_coverage_by_country_clean.csv']


In [28]:
# --- Shared helper ---

NULL_STRINGS = {"na", "n/a", "nodata", "no data", "", "none", "-", ".."}

def normalize_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Replace common string null representations with actual NaN."""
    return df.replace(
        {v: pd.NA for v in NULL_STRINGS | {s.title() for s in NULL_STRINGS} | {s.upper() for s in NULL_STRINGS}}
    )


def extract_leading_number(val) -> float | None:
    """
    Extract the leading number from a string like '3500[3000-4400]' or '500[500-500]'.
    Returns the number as float, or None if not parseable.
    """
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    match = re.match(r'^([\d,\.]+)', val_str)
    if match:
        return float(match.group(1).replace(",", ""))
    return None

In [29]:
# --- Schema A: no_of_* files ---

def clean_hiv_schema_a(path: Path, eu_countries: set[str]) -> pd.DataFrame:
    """
    Clean a Schema A HIV file (no_of_* pattern).
    - Drop 'Count' (raw string with embedded ranges)
    - Keep Count_median as the canonical value
    - Normalize null strings
    - Filter to EU countries
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape  : {df.shape}")
    print(f"  Columns    : {list(df.columns)}")

    # Drop raw 'Count' column — median/min/max columns already exist
    if "Count" in df.columns:
        df = df.drop(columns=["Count"])
        print(f"  Dropped    : 'Count'")

    # Normalize null strings
    df = normalize_nulls(df)

    # Cast numeric columns
    for col in ["Count_median", "Count_min", "Count_max"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filter to EU countries
    before = len(df)
    df = df[df["Country"].isin(eu_countries)]
    print(f"  EU filter  : {before} → {len(df)} rows")

    return df


schema_a_cleaned: dict[str, pd.DataFrame] = {}

for f in schema_a_files:
    schema_a_cleaned[f.stem] = clean_hiv_schema_a(f, EU_COUNTRIES)


no_of_cases_adults_15_to_49_by_country_clean.csv
  Raw shape  : (680, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 680 → 104 rows

no_of_deaths_by_country_clean.csv
  Raw shape  : (510, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 510 → 78 rows

no_of_people_living_with_hiv_by_country_clean.csv
  Raw shape  : (680, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 680 → 104 rows


In [30]:
# Inspect Schema A sample
key = list(schema_a_cleaned.keys())[0]
print(f"Sample from: {key}")
schema_a_cleaned[key].head(6)

Sample from: no_of_cases_adults_15_to_49_by_country_clean


,Country,Year,Count_median,Count_min,Count_max,WHO Region
7,Austria,2018,NaN,NaN,NaN,Europe
14,Belgium,2018,NaN,NaN,NaN,Europe
23,Bulgaria,2018,0.1,0.1,0.1,Europe
39,Croatia,2018,0.1,0.1,0.1,Europe
41,Cyprus,2018,NaN,NaN,NaN,Europe
45,Denmark,2018,0.1,0.1,0.1,Europe


In [31]:
# --- Schema B: art_coverage_* files ---

def clean_hiv_schema_b(path: Path, eu_countries: set[str]) -> pd.DataFrame:
    """
    Clean a Schema B HIV file (art_coverage / art_pediatric pattern).
    - No 'Year' column — this is a snapshot dataset
    - Normalize 'Nodata' / 'No data' → NaN
    - Parse bracketed ranges like '500[500-500]' → extract leading number
    - Filter to EU countries
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape  : {df.shape}")
    print(f"  Columns    : {list(df.columns)}")

    # Normalize null strings first
    df = normalize_nulls(df)

    # Parse bracketed ranges in all non-Country, non-WHO columns
    skip_cols = {"Country", "WHO Region"}
    for col in df.columns:
        if col in skip_cols:
            continue
        if df[col].dtype == object:
            has_brackets = df[col].dropna().astype(str).str.contains(r'\[', regex=True).any()
            if has_brackets:
                df[col] = df[col].apply(extract_leading_number)
                print(f"  Parsed brackets in: '{col}'")
            else:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filter to EU countries
    before = len(df)
    df = df[df["Country"].isin(eu_countries)]
    print(f"  EU filter  : {before} → {len(df)} rows")

    return df


schema_b_cleaned: dict[str, pd.DataFrame] = {}

for f in schema_b_files:
    schema_b_cleaned[f.stem] = clean_hiv_schema_b(f, EU_COUNTRIES)


art_coverage_by_country_clean.csv
  Raw shape  : (170, 11)
  Columns    : ['Country', 'Reported number of people receiving ART', 'Estimated number of people living with HIV', 'Estimated ART coverage among people living with HIV (%)', 'Estimated number of people living with HIV_median', 'Estimated number of people living with HIV_min', 'Estimated number of people living with HIV_max', 'Estimated ART coverage among people living with HIV (%)_median', 'Estimated ART coverage among people living with HIV (%)_min', 'Estimated ART coverage among people living with HIV (%)_max', 'WHO Region']
  EU filter  : 170 → 26 rows

art_pediatric_coverage_by_country_clean.csv
  Raw shape  : (170, 11)
  Columns    : ['Country', 'Reported number of children receiving ART', 'Estimated number of children needing ART based on WHO methods', 'Estimated ART coverage among children (%)', 'Estimated number of children needing ART based on WHO methods_median', 'Estimated number of children needing ART based on WH

In [32]:
# Inspect Schema B sample
key = list(schema_b_cleaned.keys())[0]
print(f"Sample from: {key}")
schema_b_cleaned[key].head(6)

Sample from: art_coverage_by_country_clean


,Country,Reported number of people receiving ART,Estimated number of people living with HIV,Estimated ART coverage among people living with HIV (%),Estimated number of people living with HIV_median,Estimated number of people living with HIV_min,Estimated number of people living with HIV_max,Estimated ART coverage among people living with HIV (%)_median,Estimated ART coverage among people living with HIV (%)_min,Estimated ART coverage among people living with HIV (%)_max,WHO Region
7,Austria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
14,Belgium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
23,Bulgaria,1500,3500[3000–4100],41[35–48],3500.0,3000.0,4100.0,41.0,35.0,48.0,Europe
39,Croatia,1200,1600[1400–1700],75[67–83],1600.0,1400.0,1700.0,75.0,67.0,83.0,Europe
41,Cyprus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
45,Denmark,5500,6200[5600–7000],89[79–95],6200.0,5600.0,7000.0,89.0,79.0,95.0,Europe


In [33]:
# Save cleaned HIV/AIDS CSVs
hiv_out_dir = CLEANED_DIR / "HIV_AIDS_data"
hiv_out_dir.mkdir(parents=True, exist_ok=True)

for key, df in {**schema_a_cleaned, **schema_b_cleaned}.items():
    out_path = hiv_out_dir / f"{key}_cleaned.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"  Saved: {out_path.name}  ({len(df)} rows)")

  Saved: no_of_cases_adults_15_to_49_by_country_clean_cleaned.csv  (104 rows)
  Saved: no_of_deaths_by_country_clean_cleaned.csv  (78 rows)
  Saved: no_of_people_living_with_hiv_by_country_clean_cleaned.csv  (104 rows)
  Saved: art_coverage_by_country_clean_cleaned.csv  (26 rows)
  Saved: art_pediatric_coverage_by_country_clean_cleaned.csv  (26 rows)
